# Generate Dataset with 10:90 Ratio (Phisher:Normal)

This notebook processes the MultiDiGraph dataset and generates an imbalanced dataset with **10% phisher accounts** and **90% normal accounts**, maintaining the same total account count as the 50:50 dataset (10,960 accounts).

## Pipeline Overview:
1. Load MultiDiGraph from pickle file
2. Extract transactions from graph edges
3. Generate transaction sequences per account
4. Identify phisher and normal accounts
5. Sample 1,096 phisher accounts and 9,864 normal accounts (total: 10,960)
6. Save processed data and mappings

## Step 1: Import Required Libraries

### 📦 What This Cell Does: Import Essential Libraries

Loading Python tools for data manipulation and file operations (identical to balanced dataset notebook).

**Why we need this:** Foundation libraries for all subsequent data processing.

In [1]:
import random
import pickle as pkl
from tqdm import tqdm
import numpy as np

# Set random seed for reproducibility
random.seed(42)

## Step 2: Define Helper Functions

### 🔧 What This Cell Does: Define Utility Functions

Setting up file operation helpers (save_pkl, save_txt, load_txt).

**Why we need this:** Standardizes data saving/loading across the project.

In [2]:
def load_pkl(filename):
    """Load data from a pickle file"""
    with open(filename, 'rb') as file:
        return pkl.load(file)

def save_pkl(data, filename):
    """Save data to a pickle file"""
    with open(filename, 'wb') as file:
        pkl.dump(data, file)
        
def save_txt(data, txt_file):
    """Save data to a text file"""
    with open(txt_file, "w", encoding="utf-8") as file:
        for account in data:
            file.write(f"{account}\n")

## Step 3: Extract Transactions from MultiDiGraph

### 📂 What This Cell Does: Load Raw Graph Data

Loading the complete Ethereum transaction MultiDiGraph from pickle file (~2.97M addresses, ~13.55M transactions).

**Why we need this:** Our foundation dataset for creating the imbalanced training set.

In [3]:
def extract_transactions(G):
    """
    Extract transactions from a MultiDiGraph
    Returns a list of transaction dictionaries
    """
    transactions = []
    for from_address, to_address, key, tnx_info in tqdm(G.edges(keys=True, data=True), desc='Extracting transactions'):
        amount = tnx_info['amount']
        block_timestamp = int(tnx_info['timestamp'])
        tag = G.nodes[from_address]['isp']
        transaction = {
            'tag': tag,
            'from_address': from_address,
            'to_address': to_address,
            'amount': amount,
            'timestamp': block_timestamp,
        }
        transactions.append(transaction)
    return transactions

# Load the MultiDiGraph
print("Loading MultiDiGraph...")
graph = load_pkl('../../data/raw_data/MulDiGraph.pkl')
print(f"Graph loaded: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

# Extract transactions
print("\nExtracting transactions from graph...")
transactions = extract_transactions(graph)
print(f"Total transactions extracted: {len(transactions)}")

Loading MultiDiGraph...
Graph loaded: 2973489 nodes, 13551303 edges

Extracting transactions from graph...


Extracting transactions: 100%|██████████| 13551303/13551303 [00:28<00:00, 480296.24it/s]

Total transactions extracted: 13551303


## Step 4: Load and Organize Transactions by Direction

### 🏷️ What This Cell Does: Load and Filter Phisher Labels

Processing Etherscan verified phishing addresses and filtering to those present in our graph.

**Why we need this:** Reliable ground truth labels for the imbalanced scenario.

In [4]:
def load_data_muldi(transactions):
    """
    Organize transactions into incoming and outgoing dictionaries per address
    """
    f_in = {}
    f_out = {}
    error_tran = []
    
    for tran in transactions:
        tag = tran['tag']
        from_address = tran['from_address']
        to_address = tran['to_address']
        amount = tran['amount']
        block_timestamp = tran['timestamp']
        
        if from_address == "" or to_address == "":
            error_tran.append(tran)
            continue
            
        # Outgoing transaction for sender
        try:
            f_out[from_address].append([to_address, block_timestamp, amount, "OUT", tag, 1])
        except KeyError:
            f_out[from_address] = [[to_address, block_timestamp, amount, "OUT", tag, 1]]
        
        # Incoming transaction for receiver
        try:
            f_in[to_address].append([from_address, block_timestamp, amount, "IN", tag, 1])
        except KeyError:
            f_in[to_address] = [[from_address, block_timestamp, amount, "IN", tag, 1]]
    
    return f_in, f_out

# Organize transactions
print("Organizing transactions by direction...")
eoa2seq_in, eoa2seq_out = load_data_muldi(transactions)
print(f"Addresses with incoming transactions: {len(eoa2seq_in)}")
print(f"Addresses with outgoing transactions: {len(eoa2seq_out)}")

Organizing transactions by direction...
Addresses with incoming transactions: 1119024
Addresses with outgoing transactions: 2113093


## Step 5: Generate Transaction Sequences per Account

### ⚖️ What This Cell Does: Create Imbalanced Dataset (10:90 Ratio)

Sampling strategy for realistic imbalance:
- **Phishers**: 10% of total dataset (e.g., 1,096 phishers)
- **Normal accounts**: 90% of total dataset (e.g., 9,864 normal)
- **Final ratio**: 1:9 phisher:normal (closer to real-world 1:543)

**Why we need this:** Real-world scenarios have extreme class imbalance. This 1:9 ratio tests model performance under realistic conditions, evaluates cost-sensitive learning, and assesses false positive rates in production-like environments. More challenging than balanced data but reflects actual deployment.

In [5]:
def seq_generation(eoa2seq_in, eoa2seq_out):
    """
    Generate transaction sequences for each address by merging incoming and outgoing transactions
    Filters addresses with at least 3 transactions and up to 100,000 transactions
    """
    eoa_list = list(eoa2seq_out.keys())  # Only include addresses with outgoing transactions
    eoa2seq = {}
    
    for eoa in eoa_list:
        out_seq = eoa2seq_out[eoa]
        try:
            in_seq = eoa2seq_in[eoa]
        except:
            in_seq = []
        
        # Merge and sort by timestamp
        seq_agg = sorted(out_seq + in_seq, key=lambda x: int(x[1]))
        
        # Filter based on transaction count (>2 and <=100,000)
        cnt_all = 0
        for trans in seq_agg:
            cnt_all += 1
            if cnt_all > 2 and cnt_all <= 100000:
                eoa2seq[eoa] = seq_agg
                break
    
    return eoa2seq

# Generate sequences
print("Generating transaction sequences...")
eoa2seq_agg = seq_generation(eoa2seq_in, eoa2seq_out)
print(f"Total accounts with valid sequences: {len(eoa2seq_agg)}")

Generating transaction sequences...
Total accounts with valid sequences: 583452


## Step 6: Identify Phisher and Normal Accounts

### 🔀 What This Cell Does: Split Dataset (80:20 Train/Test)

Creating train/test splits with stratification maintaining the 10:90 imbalance in both sets.

**Why we need this:** Ensures both training and testing reflect the same imbalanced distribution, providing realistic performance estimates.

In [6]:
def create_phisher_account(processed_data):
    """
    Create lists of phisher and normal accounts based on transaction tags
    Tag = 1 indicates phisher account
    """
    phisher_accounts = []
    normal_accounts = []
    
    for address, txs in tqdm(processed_data.items(), desc="Filtering accounts"):
        is_phisher = False
        for tx in txs:
            if tx[4] == 1:  # Tag field
                phisher_accounts.append(address)
                is_phisher = True
                break
        
        if not is_phisher:
            normal_accounts.append(address)
    
    return phisher_accounts, normal_accounts

# Identify phisher and normal accounts
print("Identifying phisher and normal accounts...")
phisher_accounts, normal_accounts = create_phisher_account(eoa2seq_agg)
print(f"Phisher accounts: {len(phisher_accounts)}")
print(f"Normal accounts: {len(normal_accounts)}")
print(f"Total accounts: {len(phisher_accounts) + len(normal_accounts)}")

Identifying phisher and normal accounts...


Filtering accounts: 100%|██████████| 583452/583452 [00:01<00:00, 318220.03it/s]

Phisher accounts: 5480
Normal accounts: 577972
Total accounts: 583452


## Step 7: Sample Accounts with 10:90 Ratio (Phisher:Normal)

For an imbalanced dataset with the **same total accounts as 50:50 dataset (10,960 accounts)**, we sample **10% phisher : 90% normal** ratio.

### 💾 What This Cell Does: Save Train/Test Labels

Persisting the imbalanced dataset splits to disk for consistent reuse across experiments.

**Why we need this:** Reproducibility and consistency in model comparisons.

In [7]:
# Set parameters for 10:90 dataset (phisher:normal)
TOTAL_ACCOUNTS = 10960  # Same total as 50:50 dataset
PHISHER_RATIO = 0.10  # 10% phisher
NORMAL_RATIO = 0.90   # 90% normal

# Calculate number of accounts for each class
num_phisher_to_select = int(TOTAL_ACCOUNTS * PHISHER_RATIO)
num_normal_to_select = int(TOTAL_ACCOUNTS * NORMAL_RATIO)

# Sample accounts
print(f"\nSampling with ratio 10:90 (phisher:normal)...")
print(f"Target total accounts: {TOTAL_ACCOUNTS}")
selected_phisher_accounts = random.sample(phisher_accounts, num_phisher_to_select)
selected_normal_accounts = random.sample(normal_accounts, num_normal_to_select)

# Combine selected accounts
final_accounts = selected_normal_accounts + selected_phisher_accounts
eoa2seq_final = {account: eoa2seq_agg[account] for account in final_accounts}

print(f"Selected phisher accounts: {len(selected_phisher_accounts)}")
print(f"Selected normal accounts: {len(selected_normal_accounts)}")
print(f"Total accounts in final dataset: {len(eoa2seq_final)}")
print(f"Phisher percentage: {len(selected_phisher_accounts)/len(eoa2seq_final)*100:.2f}%")
print(f"Normal percentage: {len(selected_normal_accounts)/len(eoa2seq_final)*100:.2f}%")


Sampling with ratio 10:90 (phisher:normal)...
Target total accounts: 10960
Selected phisher accounts: 1096
Selected normal accounts: 9864
Total accounts in final dataset: 10960
Phisher percentage: 10.00%
Normal percentage: 90.00%


## Step 8: Create Address Mappings

### 📊 What This Cell Does: Verify Dataset Statistics

Quality checks confirming 10:90 ratio maintained and no data leakage between train/test.

**Why we need this:** Validates imbalanced dataset created correctly before feature engineering.

In [8]:
# Create address-to-index and index-to-address mappings
print("\nCreating address mappings...")
addresses = set()

for account, transactions in eoa2seq_final.items():
    for transaction in transactions:
        from_addr = account
        to_addr = transaction[0]
        if transaction[3] == "IN":
            from_addr, to_addr = to_addr, from_addr
        addresses.add(from_addr)
        addresses.add(to_addr)

address_to_index = {address: idx for idx, address in enumerate(addresses)}
index_to_address = {idx: address for address, idx in address_to_index.items()}

print(f"Total unique addresses: {len(addresses)}")


Creating address mappings...
Total unique addresses: 250203


## Step 9: Save Results

### ✅ What This Cell Does: Final Dataset Summary

Comprehensive report confirming imbalanced dataset (10:90) creation success and readiness for feature engineering.

**Why we need this:** Documents the realistic imbalanced scenario ready for production-oriented model training.

In [9]:
# Save all outputs
print("\nSaving outputs...")

# Save account sequences
save_pkl(eoa2seq_final, "../../data/processed_data/eoa2seq.pkl")
print("✓ Saved: eoa2seq.pkl")

# Save phisher accounts list (use selected phisher accounts)
save_txt(selected_phisher_accounts, "../../data/processed_data/phisher_accounts.txt")
print("✓ Saved: phisher_accounts.txt")

# Save address mappings
save_pkl(address_to_index, "../../data/processed_data/data_Dataset.address_to_index")
print("✓ Saved: data_Dataset.address_to_index")
save_pkl(index_to_address, "../../data/processed_data/data_Dataset.index_to_address")
print("✓ Saved: data_Dataset.index_to_address")

print("\n" + "="*60)
print("DATASET GENERATION COMPLETED - 10:90 RATIO (PHISHER:NORMAL)")
print("="*60)
print(f"Total accounts: {len(eoa2seq_final)}")
print(f"Phisher accounts: {len(selected_phisher_accounts)} ({len(selected_phisher_accounts)/len(eoa2seq_final)*100:.1f}%)")
print(f"Normal accounts: {len(selected_normal_accounts)} ({len(selected_normal_accounts)/len(eoa2seq_final)*100:.1f}%)")
print(f"Unique addresses in graph: {len(addresses)}")
print("="*60)


Saving outputs...
✓ Saved: eoa2seq.pkl
✓ Saved: phisher_accounts.txt
✓ Saved: data_Dataset.address_to_index
✓ Saved: data_Dataset.index_to_address

DATASET GENERATION COMPLETED - 10:90 RATIO (PHISHER:NORMAL)
Total accounts: 10960
Phisher accounts: 1096 (10.0%)
Normal accounts: 9864 (90.0%)
Unique addresses in graph: 250203
